# Shift-weighted conformal prediction: extension to IPD Brain

`shift_weighted_conformal.ipynb` tests whether covariate-shift-weighted conformal
prediction restores formal coverage under a leave-institution-out split within TCGA —
a real but comparatively mild shift (same country, same broad acquisition pipeline).
This notebook asks the harder question: does the same correction work under a
genuinely cross-continent shift, using IPD Brain as the evaluation cohort instead of
TCGA's own held-out sites?

## 0. Config, imports, statistical helpers

Helpers are copied verbatim from `shift_weighted_conformal.ipynb`: the
weighted-quantile formula, domain-classifier construction, ESS diagnostic, and the selective-breakdown / full-metric-panel /
matched-random-deferral-null machinery from its Parts 8b/9/9b — so numbers stay
comparable in method throughout.

In [ ]:
from __future__ import annotations

import hashlib
import os
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

from IPython.display import display

RNG_SEED = 42
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 60)

if Path.cwd().name == "scripts":
    os.chdir("..")
BASE = Path.cwd().resolve()
print("Project root:", BASE)

OUT_DIR = BASE / "outputs/shift_weighted_conformal_ipd_brain_extension"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TCGA_MASTER_CSV = BASE / "outputs/selective_prediction_bench/master_patient_table.csv"
IPD_MASTER_CSV = BASE / "outputs/ipd_brain_baseline_bench/master_patient_table.csv"

SITE_COL = "tissue_source_site.name"
TARGET_COVERAGES = (0.90, 0.80, 0.70)
ALPHAS = [round(1 - c, 2) for c in TARGET_COVERAGES]

MIN_SITE_N = 15
SITE_PREVALENCE_BAND = (0.25, 0.75)
MIN_CAL_N, MAX_CAL_FRAC = 100, 0.35
K_RANGE = (2, 3, 4)

N_BOOT = 2000
N_RANDOM_DEFERRAL = 1000
UNI2_FEATURES = ["ood_topk_uni2", "ood_attention_uni2", "ood_mean_uni2"]
HOPT_FEATURES = ["ood_topk_hoptimus", "ood_attention_hoptimus", "ood_mean_hoptimus"]
PREDICTIVE_UNCERTAINTY_FEATURES = ["ensemble_entropy", "total_mi", "mean_within_encoder_mi",
                                    "between_encoder_mi", "aleatoric"]
FULL_FEATURES = UNI2_FEATURES + HOPT_FEATURES + PREDICTIVE_UNCERTAINTY_FEATURES
LEAD_METRICS = ["auroc", "sensitivity", "specificity", "ece", "cal_gap", "bss"]


WEIGHT_SCORE_SPLIT_SEED = 42
A_WEIGHT_FRACTION = 0.25   # fraction of Level A allocated to weight-fitting (age-stratified)

print("Config loaded.")


Project root: /cs/student/project_msc/2025/aibh/mpapageo
Config loaded.


In [ ]:
def stable_seed(*parts):
    digest = hashlib.md5(repr(parts).encode()).hexdigest()
    return int(digest, 16) % 9999


def bh_fdr(p):
    p = np.asarray(p, float); n = len(p); order = np.argsort(p)
    q = p[order] * n / (np.arange(1, n + 1))
    q = np.clip(np.minimum.accumulate(q[::-1])[::-1], 0, 1)
    out = np.empty(n); out[order] = q
    return out


def lac_score(p_true):
    return 1 - np.asarray(p_true, float)


def weighted_conformal_qhats(cal_scores, cal_w, test_w, alpha):
    order = np.argsort(cal_scores)
    s_sorted = np.asarray(cal_scores)[order]
    w_sorted = np.asarray(cal_w)[order]
    cumsum = np.cumsum(w_sorted)
    total_cal_w = cumsum[-1]
    targets = (1 - alpha) * (total_cal_w + np.asarray(test_w))
    idx = np.searchsorted(cumsum, targets, side="left")
    q_hat = np.where(idx < len(s_sorted), s_sorted[np.clip(idx, 0, len(s_sorted) - 1)], np.inf)
    return q_hat


def prediction_sets(prob_mutant, q_hat):
    p1 = np.asarray(prob_mutant, float)
    in1 = ((1 - p1) <= q_hat).astype(int)
    in0 = (p1 <= q_hat).astype(int)
    return in0, in1


def sets_for(eval_df_, cal_scores, cal_w, test_w, alpha):
    q_hat = weighted_conformal_qhats(cal_scores, cal_w, test_w, alpha)
    return prediction_sets(eval_df_["prob_mutant"].to_numpy(), q_hat)


def covered_bool(eval_df_, cal_scores, cal_w, test_w, alpha):
    y = eval_df_["label"].to_numpy()
    in0, in1 = sets_for(eval_df_, cal_scores, cal_w, test_w, alpha)
    return np.where(y == 1, in1, in0).astype(bool)


def coverage_bootstrap_ci(covered, n_boot=N_BOOT, seed=0):
    covered = np.asarray(covered, dtype=float)
    n = len(covered)
    if n == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_means = covered[idx].mean(axis=1)
    return float(np.percentile(boot_means, 2.5)), float(np.percentile(boot_means, 97.5))


def paired_diff_test(covered_a, covered_b, n_boot=N_BOOT, seed=0):
    """Paired bootstrap test of coverage(a) - coverage(b) on the same eval patients."""
    a, b = np.asarray(covered_a, float), np.asarray(covered_b, float)
    n = len(a)
    obs = float(a.mean() - b.mean())
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, size=(n_boot, n))
    diffs = a[idx].mean(axis=1) - b[idx].mean(axis=1)
    lo, hi = float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))
    p = min(1.0, 2 * min((np.sum(diffs <= 0) + 1) / (n_boot + 1),
                          (np.sum(diffs >= 0) + 1) / (n_boot + 1)))
    return obs, lo, hi, float(p)


def effective_sample_size(w):
    w = np.asarray(w, float)
    return float(w.sum() ** 2 / np.sum(w ** 2))


def build_weights_leakfree(cal_df_, eval_weight_df_, eval_score_df_, features):
    train_df = pd.concat(
        [cal_df_[features].assign(_domain=0), eval_weight_df_[features].assign(_domain=1)],
        ignore_index=True,
    ).dropna(subset=features).reset_index(drop=True)
    X_train = train_df[features].to_numpy(float)
    y_train = train_df["_domain"].to_numpy(int)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG_SEED)
    oof = cross_val_predict(GaussianNB(), X_train, y_train, cv=cv, method="predict_proba")[:, 1]
    auroc = roc_auc_score(y_train, oof)
    n_cal = int((train_df["_domain"] == 0).sum())
    p_cal_oof = np.clip(oof[:n_cal], 1e-2, 1 - 1e-2)
    cal_w = p_cal_oof / (1 - p_cal_oof)

    clf_final = GaussianNB().fit(X_train, y_train)
    X_score = eval_score_df_[features].to_numpy(float)
    p_score = np.clip(clf_final.predict_proba(X_score)[:, 1], 1e-2, 1 - 1e-2)
    score_w = p_score / (1 - p_score)

    lo, hi = np.quantile(cal_w, 0.01), np.quantile(cal_w, 0.99)
    return np.clip(cal_w, lo, hi), np.clip(score_w, lo, hi), float(auroc)


def build_siteholdout_weights_leakfree(cal_df_, eval_weight_df_, eval_score_df_, features, cal_sites):
    site_col = cal_df_["site"].to_numpy()
    cal_w = np.zeros(len(cal_df_))
    fold_aurocs = []
    for held_out in sorted(cal_sites):
        train_mask = site_col != held_out
        test_mask = site_col == held_out
        X_train = np.vstack([
            cal_df_.loc[train_mask, features].to_numpy(float),
            eval_weight_df_[features].to_numpy(float),
        ])
        y_train = np.concatenate([np.ones(int(train_mask.sum())), np.zeros(len(eval_weight_df_))])
        clf = GaussianNB().fit(X_train, y_train)
        fold_aurocs.append(roc_auc_score(y_train, clf.predict_proba(X_train)[:, 1]))
        X_test = cal_df_.loc[test_mask, features].to_numpy(float)
        p_test = np.clip(clf.predict_proba(X_test)[:, 1], 1e-2, 1 - 1e-2)
        # y_train uses 1=calibration, 0=eval_weight -- so p_test = P(cal|x), and the
        # odds ratio needed is P(eval|x)/P(cal|x) = (1-p_test)/p_test.
        cal_w[test_mask] = (1 - p_test) / p_test

    X_full = np.vstack([cal_df_[features].to_numpy(float), eval_weight_df_[features].to_numpy(float)])
    y_full = np.concatenate([np.zeros(len(cal_df_)), np.ones(len(eval_weight_df_))])
    clf_full = GaussianNB().fit(X_full, y_full)
    X_score = eval_score_df_[features].to_numpy(float)
    p_score = np.clip(clf_full.predict_proba(X_score)[:, 1], 1e-2, 1 - 1e-2)
    score_w = p_score / (1 - p_score)

    lo, hi = np.quantile(cal_w, 0.01), np.quantile(cal_w, 0.99)
    return np.clip(cal_w, lo, hi), np.clip(score_w, lo, hi), float(np.mean(fold_aurocs))


def expected_calibration_error(y, p, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    idx_bin = np.clip(np.digitize(p, edges) - 1, 0, n_bins - 1)
    ece, n = 0.0, len(y)
    for b in range(n_bins):
        m = idx_bin == b
        if m.any():
            ece += m.sum() / n * abs(y[m].mean() - p[m].mean())
    return ece


def metric_suite(y, p, thr):
    y = np.asarray(y, int); p = np.asarray(p, float)
    out = {"n": len(y)}
    if len(np.unique(y)) < 2:
        return {**out, **{m: np.nan for m in LEAD_METRICS}}
    yhat = (p >= thr).astype(int)
    tp = int(((yhat == 1) & (y == 1)).sum()); fn = int(((yhat == 0) & (y == 1)).sum())
    tn = int(((yhat == 0) & (y == 0)).sum()); fp = int(((yhat == 1) & (y == 0)).sum())
    out["auroc"] = roc_auc_score(y, p)
    out["sensitivity"] = tp / max(1, tp + fn)
    out["specificity"] = tn / max(1, tn + fp)
    out["ece"] = expected_calibration_error(y, p)
    out["cal_gap"] = float(p.mean() - y.mean())
    prevalence_variance = y.mean() * (1 - y.mean())
    out["bss"] = 1 - np.mean((p - y) ** 2) / prevalence_variance if prevalence_variance > 1e-6 else np.nan
    return out


def perm_p_two_sided(observed, null_values):
    null_values = np.asarray(null_values, float)
    null_values = null_values[np.isfinite(null_values)]
    n = len(null_values)
    if n == 0 or not np.isfinite(observed):
        return np.nan
    p_hi = (np.sum(null_values >= observed) + 1) / (n + 1)
    p_lo = (np.sum(null_values <= observed) + 1) / (n + 1)
    return float(min(1.0, 2 * min(p_hi, p_lo)))


def random_deferral_null(y, p, thr, n_keep, n_rep=N_RANDOM_DEFERRAL, seed=0):
    y = np.asarray(y, int); p = np.asarray(p, float)
    n = len(y)
    draws = {m: np.full(n_rep, np.nan) for m in LEAD_METRICS}
    if n_keep <= 0 or n_keep > n:
        return draws
    rng = np.random.default_rng(seed)
    for r in range(n_rep):
        keep = np.zeros(n, dtype=bool)
        keep[rng.choice(n, n_keep, replace=False)] = True
        m = metric_suite(y[keep], p[keep], thr)
        for metric in LEAD_METRICS:
            draws[metric][r] = m[metric]
    return draws


print("Helpers defined.")

Helpers defined.


## 1. Calibration set (TCGA) - reproduced deterministically, not reselected

Reuses the exact same site-selection search as `shift_weighted_conformal.ipynb` Part
2, so `cal_df` here is identical

In [47]:
master = pd.read_csv(TCGA_MASTER_CSV)
df = master.copy()
df["site"] = df[SITE_COL].fillna("Unknown").astype(str).str.strip()
df = df.dropna(subset=["label", "prob_mutant", "site"]).reset_index(drop=True)
df["label"] = df["label"].astype(int)
OVERALL_PREVALENCE = float(df["label"].mean())

site_table = df.groupby("site")["label"].agg(n="size", prevalence="mean").sort_values("n", ascending=False)
candidates = site_table[
    (site_table["n"] >= MIN_SITE_N) & (site_table["prevalence"].between(*SITE_PREVALENCE_BAND))
].index.tolist()

rows = []
for k in K_RANGE:
    for combo in combinations(candidates, k):
        n = int(site_table.loc[list(combo), "n"].sum())
        if not (MIN_CAL_N <= n <= MAX_CAL_FRAC * len(df)):
            continue
        prev = float((site_table.loc[list(combo), "n"] * site_table.loc[list(combo), "prevalence"]).sum() / n)
        max_skew = float((site_table.loc[list(combo), "prevalence"] - 0.5).abs().max())
        rows.append({"sites": combo, "k": k, "n": n, "combined_prevalence": round(prev, 3),
                     "combined_skew": round(abs(prev - OVERALL_PREVALENCE), 4),
                     "max_individual_site_skew": round(max_skew, 3)})

shortlist = (pd.DataFrame(rows)
             .sort_values(["combined_skew", "max_individual_site_skew", "n"], ascending=[True, True, False])
             .reset_index(drop=True))
NEW_CALIBRATION_SITES = set(shortlist.iloc[0]["sites"])

cal_mask = df["site"].isin(NEW_CALIBRATION_SITES)
cal_df = df.loc[cal_mask].copy().reset_index(drop=True)

print(f"Calibration sites: {NEW_CALIBRATION_SITES}")
print(f"cal_df: n={len(cal_df)}, prevalence={cal_df['label'].mean():.3f}")
assert len(cal_df) == 115, f"expected the same n=115 calibration set as the original notebook, got {len(cal_df)}"
print("Matches the n=115 calibration set behind the existing RQ2b numbers.")

Calibration sites: {'University of Florida', 'Thomas Jefferson University', 'Case Western'}
cal_df: n=115, prevalence=0.548
Matches the n=115 calibration set behind the existing RQ2b numbers.


## 2. Evaluation set (IPD Brain) 

Loads the already-built, already-merged master table and splits into Level A (the
only scored population).

In [ ]:
ipd = pd.read_csv(IPD_MASTER_CSV).rename(columns={"patient": "patient_id"})
print(f"IPD Brain master table: n={len(ipd)}, {len(ipd.columns)} columns")

required_cols = (["prob_mutant", "label", "level", "age_band"]
                  + UNI2_FEATURES + HOPT_FEATURES + PREDICTIVE_UNCERTAINTY_FEATURES)
missing = [c for c in required_cols if c not in ipd.columns]
assert not missing, f"IPD Brain master table is missing required columns: {missing}"

eval_A = ipd[ipd["level"] == "A"].reset_index(drop=True)
eval_C = ipd[ipd["level"] == "C"].reset_index(drop=True)

YOUNG_BANDS = {"<20", "20-39", "40-54"}
young_wt_mask = (eval_A["label"] == 0) & (eval_A["age_band"].isin(YOUNG_BANDS))
eval_young_wt = eval_A[young_wt_mask].reset_index(drop=True)
eval_A_scorable = eval_A[~young_wt_mask].reset_index(drop=True)

print(f"Level A: n={len(eval_A)}, prevalence={eval_A['label'].mean():.3f}")
print(f"  of which age<55 + wildtype (weight-fitting-only): n={len(eval_young_wt)}")
print(f"  remaining, scorable: n={len(eval_A_scorable)}, prevalence={eval_A_scorable['label'].mean():.3f}")
print(f"Level C (n={len(eval_C)}): excluded from scoring everywhere, "
      f"available for label-free weight-fitting (Section 2b)")

IPD Brain master table: n=320, 91 columns
Level A: n=312, prevalence=0.516
  of which age<55 + wildtype (weight-fitting-only): n=79
  remaining, scorable: n=233, prevalence=0.691
Level C (n=8): excluded from scoring everywhere, available for label-free weight-fitting (Section 2b)


## 2b. Weight/score split: keeps every classifier fit disjoint from the coverage it's evaluated on

Sections 3-8 fit a domain classifier to estimate covariate-shift weights (TCGA
`cal_df` vs. an IPD Brain pool). 

Free extra weight-fitting data: Level C (n=8) and the age<55 wildtype-labelled
patients (n=79, `eval_young_wt`, any grade) both have labels too unreliable to trust for scoring, but the domain classifier never looks at the label, only the OOD/covariate features. So both go entirely into the weight-fitting pool for free, at zero cost to the scorable population's evaluation power.

The split itself: age-band-stratified 25/75 split of `eval_A_scorable`
(`A_WEIGHT_FRACTION`, n=233, Level A minus the age<55-wildtype carve-out). 25%, drawn
proportionally from every remaining age band, joins `eval_young_wt` and Level C in
the weight-fitting pool; the remaining 75% is `eval_score_df`, which never touches
classifier fitting. Stratifying by age band rather than a plain random split keeps
the weight-fitting slice representative of the scorable population's age range.

In [ ]:
def stratified_weight_score_split(df_, strat_col, weight_frac, seed):
    rng = np.random.default_rng(seed)
    weight_idx = []
    for _, grp in df_.groupby(strat_col):
        n_take = min(len(grp), max(1, round(len(grp) * weight_frac)))
        weight_idx.extend(rng.choice(grp.index, size=n_take, replace=False))
    weight_df = df_.loc[weight_idx].copy().reset_index(drop=True)
    score_df = df_.drop(index=weight_idx).copy().reset_index(drop=True)
    return weight_df, score_df


A_weight_slice, eval_score_df = stratified_weight_score_split(
    eval_A_scorable, "age_band", A_WEIGHT_FRACTION, seed=WEIGHT_SCORE_SPLIT_SEED)
eval_weight_df = pd.concat([A_weight_slice, eval_young_wt, eval_C], ignore_index=True)

overlap = set(eval_weight_df["patient_id"]) & set(eval_score_df["patient_id"])
assert len(overlap) == 0, f"weight/score overlap: {overlap}"

print(f"eval_weight_df = {len(A_weight_slice)} (age-stratified slice of scorable Level A) + "
      f"{len(eval_young_wt)} (age<55 wildtype) + {len(eval_C)} (Level C) = {len(eval_weight_df)}")
print(f"eval_score_df  = {len(eval_score_df)} (remaining scorable Level A, never touches classifier fitting)")
print("\nAge-band composition (weight slice vs. eval_score_df):")
display(pd.DataFrame({
    "A_weight_slice": A_weight_slice["age_band"].value_counts(),
    "eval_score_df": eval_score_df["age_band"].value_counts(),
}).fillna(0).astype(int))

eval_weight_df = 58 (age-stratified slice of scorable Level A) + 79 (age<55 wildtype) + 8 (Level C) = 145
eval_score_df  = 175 (remaining scorable Level A, never touches classifier fitting)

Age-band composition (weight slice vs. eval_score_df):


,A_weight_slice,eval_score_df
age_band,,
20-39,20,62
40-54,14,44
55-69,20,61
70+,3,8
<20,1,0


## 3. Plain conformal, then weighted conformal -- UNI2-only vs. H-optimus-only

Runs plain split conformal (no shift correction) as the baseline, then two
independent covariate-shift-weighted versions: one built entirely from UNI2's flow
OOD features, one entirely from H-optimus's. Coverage for every scheme is evaluated
on `eval_score_df` (Level A, disjoint from whatever fit that scheme's domain
classifier, Section 2b). Every coverage estimate gets a bootstrap CI; every weighted
scheme gets a paired bootstrap test against plain, and its own effective sample
size.

In [ ]:
cal_scores_true = lac_score(np.where(cal_df["label"] == 1, cal_df["prob_mutant"], 1 - cal_df["prob_mutant"]))

# Single tier now (Level A, n=312) -- the old "Level A+B" sensitivity tier no longer exists
TIER_NAME = "Level A"
ewdf, esdf = eval_weight_df, eval_score_df

results, ess_rows = [], []
WEIGHTS_BY_TIER_SCHEME = {}

plain_w_cal = np.ones(len(cal_df))
scheme_weights = {}
for scheme_name, feats in [("weighted_uni2", UNI2_FEATURES), ("weighted_hoptimus", HOPT_FEATURES)]:
    cal_w, score_w, domain_auroc = build_weights_leakfree(cal_df, ewdf, esdf, feats)
    scheme_weights[scheme_name] = (cal_w, score_w, domain_auroc)
    ess_val = effective_sample_size(cal_w)
    ess_rows.append({"tier": TIER_NAME, "scheme": scheme_name, "domain_auroc": domain_auroc,
                      "n_cal": len(cal_w), "ess": round(ess_val, 1),
                      "ess_fraction": round(ess_val / len(cal_w), 3)})
WEIGHTS_BY_TIER_SCHEME[TIER_NAME] = scheme_weights

for cov, alpha in zip(TARGET_COVERAGES, ALPHAS):
    covered_plain = covered_bool(esdf, cal_scores_true, plain_w_cal, np.ones(len(esdf)), alpha)
    ci_lo, ci_hi = coverage_bootstrap_ci(covered_plain, seed=stable_seed("plain", TIER_NAME, cov))
    results.append({"tier": TIER_NAME, "method": "plain", "target_coverage": cov,
                     "n": len(esdf), "coverage": float(covered_plain.mean()),
                     "ci_lo": ci_lo, "ci_hi": ci_hi,
                     "delta_vs_plain": np.nan, "delta_ci_lo": np.nan, "delta_ci_hi": np.nan, "p": np.nan})

    for scheme_name in ["weighted_uni2", "weighted_hoptimus"]:
        cal_w, score_w, domain_auroc = scheme_weights[scheme_name]
        covered_w = covered_bool(esdf, cal_scores_true, cal_w, score_w, alpha)
        ci_lo, ci_hi = coverage_bootstrap_ci(covered_w, seed=stable_seed(scheme_name, TIER_NAME, cov))
        obs, dlo, dhi, p = paired_diff_test(covered_w, covered_plain,
                                             seed=stable_seed("diff", scheme_name, TIER_NAME, cov))
        results.append({"tier": TIER_NAME, "method": scheme_name, "target_coverage": cov,
                         "n": len(esdf), "coverage": float(covered_w.mean()),
                         "ci_lo": ci_lo, "ci_hi": ci_hi,
                         "delta_vs_plain": obs, "delta_ci_lo": dlo, "delta_ci_hi": dhi, "p": p})

results_df = pd.DataFrame(results)
results_df.to_csv(OUT_DIR / "coverage_results.csv", index=False)
ess_df = pd.DataFrame(ess_rows)
ess_df.to_csv(OUT_DIR / "domain_classifier_ess.csv", index=False)

print("=== Domain classifier diagnostics ===")
display(ess_df.round(3))
print("\n=== Coverage results ===")
display(results_df.round(4))

=== Domain classifier diagnostics ===


,tier,scheme,domain_auroc,n_cal,ess,ess_fraction
0,Level A,weighted_uni2,0.747,115,46.5,0.405
1,Level A,weighted_hoptimus,0.856,115,13.9,0.121



=== Coverage results ===


,tier,method,target_coverage,n,coverage,ci_lo,ci_hi,delta_vs_plain,delta_ci_lo,delta_ci_hi,p
0,Level A,plain,0.9,175,0.8743,0.8229,0.9200,NaN,NaN,NaN,NaN
1,Level A,weighted_uni2,0.9,175,0.8514,0.8000,0.9029,-0.0229,-0.0457,-0.0057,0.038
2,Level A,weighted_hoptimus,0.9,175,0.9200,0.8800,0.9543,0.0457,0.0171,0.0800,0.001
3,Level A,plain,0.8,175,0.7657,0.7029,0.8286,NaN,NaN,NaN,NaN
4,Level A,weighted_uni2,0.8,175,0.6057,0.5314,0.6800,-0.1600,-0.2171,-0.1086,0.001
5,Level A,weighted_hoptimus,0.8,175,0.8743,0.8229,0.9200,0.1086,0.0629,0.1543,0.001
6,Level A,plain,0.7,175,0.6343,0.5600,0.7029,NaN,NaN,NaN,NaN
7,Level A,weighted_uni2,0.7,175,0.5029,0.4343,0.5771,-0.1314,-0.1829,-0.0800,0.001
8,Level A,weighted_hoptimus,0.7,175,0.8057,0.7486,0.8629,0.1714,0.1200,0.2286,0.001


## 4. Full-feature domain classifier (flow OOD, both encoders + predictive-uncertainty signals)

Does mixing in the ABMIL predictor's own confidence/disagreement signals (`ensemble_entropy`, `total_mi`,
`mean_within_encoder_mi`, `between_encoder_mi`, `aleatoric`) on top of both encoders' flow OOD features change anything, or make the UNI2/H-optimus split in Section 3 worse, better, or irrelevant? Restricted to Level A (primary) from here on, using `eval_weight_df`/`eval_score_df` from Section 2b.

In [51]:
cal_w_full, score_w_full, domain_auroc_full = build_weights_leakfree(
    cal_df, eval_weight_df, eval_score_df, FULL_FEATURES)
ess_full = effective_sample_size(cal_w_full)
print(f"Full-feature domain classifier AUROC: {domain_auroc_full:.3f}")
print(f"ESS: {ess_full:.1f} / {len(cal_w_full)} ({ess_full/len(cal_w_full):.1%})")

fullfeat_rows = []
plain_w = np.ones(len(cal_df))
for cov, alpha in zip(TARGET_COVERAGES, ALPHAS):
    covered_plain = covered_bool(eval_score_df, cal_scores_true, plain_w, np.ones(len(eval_score_df)), alpha)
    covered_full = covered_bool(eval_score_df, cal_scores_true, cal_w_full, score_w_full, alpha)
    ci_lo, ci_hi = coverage_bootstrap_ci(covered_full, seed=stable_seed("fullfeat", cov))
    obs, dlo, dhi, p = paired_diff_test(covered_full, covered_plain, seed=stable_seed("diff_fullfeat", cov))
    fullfeat_rows.append({"target_coverage": cov, "coverage": float(covered_full.mean()),
                           "ci_lo": ci_lo, "ci_hi": ci_hi,
                           "delta_vs_plain": obs, "delta_ci_lo": dlo, "delta_ci_hi": dhi, "p": p})

fullfeat_df = pd.DataFrame(fullfeat_rows)
fullfeat_df.to_csv(OUT_DIR / "fullfeature_coverage.csv", index=False)
display(fullfeat_df.round(4))

Full-feature domain classifier AUROC: 0.892
ESS: 18.7 / 115 (16.3%)


,target_coverage,coverage,ci_lo,ci_hi,delta_vs_plain,delta_ci_lo,delta_ci_hi,p
0,0.9,0.9657,0.9371,0.9886,0.0914,0.0514,0.1371,0.001
1,0.8,0.7486,0.6856,0.8171,-0.0171,-0.0400,0.0000,0.099
2,0.7,0.5429,0.4686,0.6171,-0.0914,-0.1371,-0.0457,0.001


## 5. Leave-one-calibration-site-out cross-fit (most conservative variant)

Replicates `shift_weighted_conformal_leakage_free.ipynb`'s own site-holdout section.
Holds out each of the 3 TCGA calibration sites in turn (train on the other 2 sites
vs. `eval_weight_df`, predict only the held-out site's weights) for both UNI2 and
H-optimus feature sets, so no calibration patient's weight is ever influenced by
other patients from their own institution. As everywhere else in this rebuild, the
IPD-side pool doing the fitting (`eval_weight_df`) stays disjoint from the one being
scored (`eval_score_df`).

In [52]:
siteholdout_rows, siteholdout_ess = [], []
SITEHOLDOUT_WEIGHTS = {}
for scheme_name, feats in [("siteholdout_uni2", UNI2_FEATURES), ("siteholdout_hoptimus", HOPT_FEATURES)]:
    cal_w_sh, score_w_sh, fold_auroc = build_siteholdout_weights_leakfree(
        cal_df, eval_weight_df, eval_score_df, feats, NEW_CALIBRATION_SITES)
    SITEHOLDOUT_WEIGHTS[scheme_name] = (cal_w_sh, score_w_sh)
    ess_sh = effective_sample_size(cal_w_sh)
    siteholdout_ess.append({"scheme": scheme_name, "mean_fold_auroc": fold_auroc,
                             "ess": round(ess_sh, 1), "ess_fraction": round(ess_sh / len(cal_w_sh), 3)})
    plain_w = np.ones(len(cal_df))
    for cov, alpha in zip(TARGET_COVERAGES, ALPHAS):
        covered_plain = covered_bool(eval_score_df, cal_scores_true, plain_w, np.ones(len(eval_score_df)), alpha)
        covered_sh = covered_bool(eval_score_df, cal_scores_true, cal_w_sh, score_w_sh, alpha)
        ci_lo, ci_hi = coverage_bootstrap_ci(covered_sh, seed=stable_seed(scheme_name, cov))
        obs, dlo, dhi, p = paired_diff_test(covered_sh, covered_plain, seed=stable_seed("diff", scheme_name, cov))
        siteholdout_rows.append({"scheme": scheme_name, "target_coverage": cov,
                                  "coverage": float(covered_sh.mean()), "ci_lo": ci_lo, "ci_hi": ci_hi,
                                  "delta_vs_plain": obs, "delta_ci_lo": dlo, "delta_ci_hi": dhi, "p": p})

siteholdout_df = pd.DataFrame(siteholdout_rows)
siteholdout_df.to_csv(OUT_DIR / "siteholdout_coverage.csv", index=False)
print("=== Site-holdout ESS diagnostics ===")
display(pd.DataFrame(siteholdout_ess).round(3))
print("\n=== Site-holdout coverage ===")
display(siteholdout_df.round(4))

=== Site-holdout ESS diagnostics ===


,scheme,mean_fold_auroc,ess,ess_fraction
0,siteholdout_uni2,0.776,55.4,0.481
1,siteholdout_hoptimus,0.869,17.0,0.147



=== Site-holdout coverage ===


,scheme,target_coverage,coverage,ci_lo,ci_hi,delta_vs_plain,delta_ci_lo,delta_ci_hi,p
0,siteholdout_uni2,0.9,0.8514,0.8000,0.9029,-0.0229,-0.0457,-0.0057,0.029
1,siteholdout_uni2,0.8,0.6629,0.5943,0.7314,-0.1029,-0.1543,-0.0571,0.001
2,siteholdout_uni2,0.7,0.5200,0.4457,0.6000,-0.1143,-0.1601,-0.0686,0.001
3,siteholdout_hoptimus,0.9,0.9429,0.9086,0.9771,0.0686,0.0343,0.1086,0.001
4,siteholdout_hoptimus,0.8,0.8743,0.8229,0.9200,0.1086,0.0686,0.1543,0.001
5,siteholdout_hoptimus,0.7,0.8114,0.7486,0.8686,0.1771,0.1257,0.2343,0.001


## 6. Baseline: no conformal, no abstention at all

Every comparison so far has been conformal-vs-conformal. This computes the simplest possible reference: the locked classifier's own 0.5-threshold call on every Level A patient, no prediction sets, no deferral, so Section 7's retained/deferred risk numbers have an explicit anchor.

In [53]:
y_all = eval_score_df["label"].to_numpy()
p_all = eval_score_df["prob_mutant"].to_numpy()
pred_all = (p_all >= 0.5).astype(int)
err_all = (pred_all != y_all).astype(int)

baseline_error = float(err_all.mean())
baseline_ci_lo, baseline_ci_hi = coverage_bootstrap_ci(err_all, seed=stable_seed("baseline_no_conformal"))
baseline_auroc = roc_auc_score(y_all, p_all)

tp = int(((pred_all == 1) & (y_all == 1)).sum())
fn = int(((pred_all == 0) & (y_all == 1)).sum())
tn = int(((pred_all == 0) & (y_all == 0)).sum())
fp = int(((pred_all == 1) & (y_all == 0)).sum())
baseline_sensitivity = tp / max(1, tp + fn)
baseline_specificity = tn / max(1, tn + fp)

print(f"No-conformal baseline (n={len(y_all)}, Level A eval_score_df, no abstention at all):")
print(f"  Error rate:  {baseline_error:.3f} [{baseline_ci_lo:.3f}-{baseline_ci_hi:.3f}]")
print(f"  AUROC:       {baseline_auroc:.3f}")
print(f"  Sensitivity: {baseline_sensitivity:.3f}   Specificity: {baseline_specificity:.3f}")

pd.DataFrame([{"n": len(y_all), "error_rate": baseline_error, "ci_lo": baseline_ci_lo, "ci_hi": baseline_ci_hi,
               "auroc": baseline_auroc, "sensitivity": baseline_sensitivity,
               "specificity": baseline_specificity}]).to_csv(OUT_DIR / "no_conformal_baseline.csv", index=False)

No-conformal baseline (n=175, Level A eval_score_df, no abstention at all):
  Error rate:  0.211 [0.154-0.274]
  AUROC:       0.914
  Sensitivity: 0.719   Specificity: 0.944


## 7. Selective prediction: singleton (retained) vs. non-singleton (deferred)

A non-singleton set is either a both set (both labels included, a genuine safe hedge that always covers by construction) or an empty set (neither label included, never covers, a real miss regardless of how confident the underlying point-prediction is). Which dominates matters as much as the raw deferral rate. Computed on `eval_score_df` for every scheme built so far: plain, weighted_uni2, weighted_hoptimus, fullfeat, siteholdout_uni2, siteholdout_hoptimus — reusing the weights already fit in Sections 3-5, never refit here.

In [54]:
METHOD_SETS = {}
plain_w_cal, plain_w_score = np.ones(len(cal_df)), np.ones(len(eval_score_df))
METHOD_SETS["plain"] = {cov: sets_for(eval_score_df, cal_scores_true, plain_w_cal, plain_w_score, alpha)
                         for cov, alpha in zip(TARGET_COVERAGES, ALPHAS)}

level_a_weights = WEIGHTS_BY_TIER_SCHEME["Level A"]
for scheme_name in ["weighted_uni2", "weighted_hoptimus"]:
    cal_w, score_w, _ = level_a_weights[scheme_name]
    METHOD_SETS[scheme_name] = {cov: sets_for(eval_score_df, cal_scores_true, cal_w, score_w, alpha)
                                 for cov, alpha in zip(TARGET_COVERAGES, ALPHAS)}

METHOD_SETS["fullfeat"] = {cov: sets_for(eval_score_df, cal_scores_true, cal_w_full, score_w_full, alpha)
                            for cov, alpha in zip(TARGET_COVERAGES, ALPHAS)}

for scheme_name, (cal_w_sh, score_w_sh) in SITEHOLDOUT_WEIGHTS.items():
    METHOD_SETS[scheme_name] = {cov: sets_for(eval_score_df, cal_scores_true, cal_w_sh, score_w_sh, alpha)
                                 for cov, alpha in zip(TARGET_COVERAGES, ALPHAS)}

METHOD_LABELS = {
    "plain": "Plain conformal", "weighted_uni2": "Weighted (UNI2)",
    "weighted_hoptimus": "Weighted (H-optimus)", "fullfeat": "Weighted (full-feature)",
    "siteholdout_uni2": "Weighted (UNI2, site-holdout)",
    "siteholdout_hoptimus": "Weighted (H-optimus, site-holdout)",
}


def selective_breakdown_table(in0, in1, sub_df, method_key, cov):
    size = in0 + in1
    y = sub_df["label"].to_numpy()
    pred = (sub_df["prob_mutant"].to_numpy() >= 0.5).astype(int)
    err = (pred != y).astype(int)
    retained, both, empty = size == 1, size == 2, size == 0

    def _risk_ci(mask, seed_key):
        if not mask.any():
            return np.nan, np.nan, np.nan
        risk = float(err[mask].mean())
        lo, hi = coverage_bootstrap_ci(err[mask], seed=stable_seed(seed_key, method_key, cov))
        return risk, lo, hi

    risk_retained, ret_lo, ret_hi = _risk_ci(retained, "risk_retained")
    risk_both, both_lo, both_hi = _risk_ci(both, "risk_both")
    risk_empty, emp_lo, emp_hi = _risk_ci(empty, "risk_empty")

    return {"target_coverage": cov, "method": method_key, "method_label": METHOD_LABELS[method_key],
            "risk_full": float(err.mean()), "n_retained": int(retained.sum()), "risk_retained": risk_retained,
            "risk_retained_ci_lo": ret_lo, "risk_retained_ci_hi": ret_hi,
            "risk_reduction_vs_no_conformal": baseline_error / risk_retained if risk_retained else np.nan,
            "n_both": int(both.sum()), "risk_both": risk_both,
            "n_empty": int(empty.sum()), "risk_empty": risk_empty,
            "defer_rate": float((~retained).mean())}


sel_rows = []
for cov in TARGET_COVERAGES:
    for method_key, sets in METHOD_SETS.items():
        in0, in1 = sets[cov]
        sel_rows.append(selective_breakdown_table(in0, in1, eval_score_df, method_key, cov))
sel_table = pd.DataFrame(sel_rows)
sel_table.to_csv(OUT_DIR / "selective_breakdown.csv", index=False)

DISPLAY_SEL_COLS = ["target_coverage", "method_label", "defer_rate", "n_retained", "risk_retained",
                     "risk_reduction_vs_no_conformal", "n_both", "risk_both", "n_empty", "risk_empty"]
display(sel_table[DISPLAY_SEL_COLS].sort_values(["target_coverage", "method_label"]).round(3))

,target_coverage,method_label,defer_rate,n_retained,risk_retained,risk_reduction_vs_no_conformal,n_both,risk_both,n_empty,risk_empty
12,0.7,Plain conformal,0.229,135,0.178,1.189,0,NaN,40,0.325
14,0.7,Weighted (H-optimus),0.046,167,0.204,1.038,8,0.375,0,NaN
17,0.7,"Weighted (H-optimus, site-holdout)",0.063,164,0.201,1.051,11,0.364,0,NaN
13,0.7,Weighted (UNI2),0.406,104,0.154,1.374,0,NaN,71,0.296
16,0.7,"Weighted (UNI2, site-holdout)",0.389,107,0.150,1.414,0,NaN,68,0.309
15,0.7,Weighted (full-feature),0.337,116,0.181,1.168,0,NaN,59,0.271
6,0.8,Plain conformal,0.040,168,0.202,1.045,0,NaN,7,0.429
8,0.8,Weighted (H-optimus),0.291,124,0.177,1.192,51,0.294,0,NaN
11,0.8,"Weighted (H-optimus, site-holdout)",0.291,124,0.177,1.192,51,0.294,0,NaN
7,0.8,Weighted (UNI2),0.257,130,0.185,1.145,0,NaN,45,0.289


## 8. Full metric panel: AUROC/calibration/etc. vs. baseline and matched random deferral

Section 7 only reports error rate. This compares retained-set performance on the full LEAD panel (AUROC,
sensitivity, specificity, ECE, cal_gap, BSS) against two references: the full-cohort no-conformal baseline (`d_full`) and a matched-random-deferral null (`d_random`, same retained count, drawn at random 1000 times). The sharper question: does the conformal-selected retained set actually beat an arbitrary same-size subset, not just look better than not deferring at all.

No per-site breakdown here or anywhere in this notebook IPD Brain is a single institution, so there's no site
substructure to stratify this by.

In [55]:
full_metrics = metric_suite(y_all, p_all, 0.5)
print(f"No-conformal full-cohort LEAD panel (baseline, n={len(y_all)}):")
for m in LEAD_METRICS:
    print(f"  {m}: {full_metrics[m]:.3f}")

panel_rows = []
for cov in TARGET_COVERAGES:
    for method_key, sets in METHOD_SETS.items():
        in0, in1 = sets[cov]
        retained = (in0 + in1) == 1
        n_keep = int(retained.sum())
        if n_keep == 0:
            continue
        y_ret, p_ret = y_all[retained], p_all[retained]
        ret_metrics = metric_suite(y_ret, p_ret, 0.5)
        null = random_deferral_null(y_all, p_all, 0.5, n_keep, seed=stable_seed("panel_null", method_key, cov))
        for metric in LEAD_METRICS:
            obs = ret_metrics[metric]
            null_vals = null[metric]
            d_full = obs - full_metrics[metric] if np.isfinite(obs) else np.nan
            d_random = obs - np.nanmean(null_vals) if np.isfinite(obs) else np.nan
            p_val = perm_p_two_sided(obs, null_vals) if np.isfinite(obs) else np.nan
            panel_rows.append({"target_coverage": cov, "method": method_key,
                                "method_label": METHOD_LABELS[method_key], "metric": metric, "n_retained": n_keep,
                                "full": full_metrics[metric], "retained": obs, "d_full": d_full,
                                "d_random": d_random, "p": p_val})

panel_table = pd.DataFrame(panel_rows)
panel_table["q"] = bh_fdr(panel_table["p"].fillna(1.0).to_numpy())
panel_table["significant_q05"] = (panel_table["q"] < 0.05) & panel_table["p"].notna()
panel_table.to_csv(OUT_DIR / "metric_panel.csv", index=False)

print(f"\nMetric panel table: {len(panel_table)} rows "
      f"({len(TARGET_COVERAGES)} targets x {len(METHOD_SETS)} methods x {len(LEAD_METRICS)} metrics)")
n_sig = int(panel_table["significant_q05"].sum())
print(f"BH-FDR q<0.05 significant cells: {n_sig}/{len(panel_table)}")

for cov in TARGET_COVERAGES:
    print(f"\n=== target={cov:.0%}: retained-set LEAD panel ===")
    display(panel_table[panel_table["target_coverage"] == cov]
            .pivot(index="method_label", columns="metric", values="retained")[LEAD_METRICS].round(3))

No-conformal full-cohort LEAD panel (baseline, n=175):
  auroc: 0.914
  sensitivity: 0.719
  specificity: 0.944
  ece: 0.228
  cal_gap: -0.228
  bss: 0.201

Metric panel table: 108 rows (3 targets x 6 methods x 6 metrics)
BH-FDR q<0.05 significant cells: 28/108

=== target=90%: retained-set LEAD panel ===


metric,auroc,sensitivity,specificity,ece,cal_gap,bss
method_label,,,,,,
Plain conformal,0.887,0.696,0.980,0.168,-0.162,0.367
Weighted (H-optimus),0.897,0.736,1.000,0.158,-0.158,0.433
"Weighted (H-optimus, site-holdout)",0.902,0.778,1.000,0.139,-0.139,0.503
Weighted (UNI2),0.904,0.702,0.980,0.187,-0.187,0.325
"Weighted (UNI2, site-holdout)",0.903,0.699,0.980,0.186,-0.186,0.325
Weighted (full-feature),0.931,0.853,0.889,0.158,-0.158,0.265



=== target=80%: retained-set LEAD panel ===


metric,auroc,sensitivity,specificity,ece,cal_gap,bss
method_label,,,,,,
Plain conformal,0.916,0.722,0.962,0.223,-0.223,0.224
Weighted (H-optimus),0.893,0.712,0.980,0.165,-0.165,0.373
"Weighted (H-optimus, site-holdout)",0.893,0.712,0.980,0.165,-0.165,0.373
Weighted (UNI2),0.894,0.705,0.981,0.173,-0.173,0.352
"Weighted (UNI2, site-holdout)",0.902,0.722,0.962,0.186,-0.181,0.331
Weighted (full-feature),0.914,0.714,0.962,0.220,-0.220,0.233



=== target=70%: retained-set LEAD panel ===


metric,auroc,sensitivity,specificity,ece,cal_gap,bss
method_label,,,,,,
Plain conformal,0.900,0.723,0.981,0.176,-0.176,0.357
Weighted (H-optimus),0.915,0.719,0.962,0.222,-0.222,0.227
"Weighted (H-optimus, site-holdout)",0.914,0.721,0.962,0.217,-0.217,0.241
Weighted (UNI2),0.886,0.724,1.000,0.153,-0.153,0.435
"Weighted (UNI2, site-holdout)",0.886,0.733,1.000,0.150,-0.150,0.447
Weighted (full-feature),0.887,0.682,1.000,0.167,-0.167,0.384


## 10. Ablation: calibrating directly on IPD Brain instead of transporting TCGA's correction

Every result above transports a correction fit on TCGA to IPD Brain. This asks the different question: if calibration and evaluation are both drawn from IPD Brain itself, with no cross-cohort shift to correct for, does plain conformal, no weighting at all, already hit nominal coverage? Split conformal's validity only requires calibration and evaluation to be exchangeable; a random split within one institution satisfies that by construction, unlike the TCGA-to-IPD-Brain split used everywhere else in this notebook.

Not a proposed alternative to the TCGA-transported correction — it needs enough labelled IPD Brain data to calibrate on before any deferral decision can be trusted there, which defeats the point of locking a correction once and deploying it to a new site with no local labels. This is a diagnostic isolating why the TCGA-calibrated schemes needed correction in the first place, not a deployable substitute for one.

50 random 50/50 splits of `eval_A_scorable`, plain (unweighted) conformal on each,
averaged.

Deliberately independent of Section 2b's weight/score split, but not of Section 2's age<55-wildtype carve-out. This section has no domain classifier, so the population-level leakage problem that motivated Section 2b doesn't apply here, it uses `eval_A_scorable` directly rather than `eval_score_df`. It does still exclude `eval_young_wt`, though: those patients' labels are being treated as too unreliable to trust for scoring everywhere else in this notebook, so letting them freely enter both the calibration and evaluation halves of a self-calibration split here (which relies on local labels directly) would be inconsistent with that. Its role is as the natural comparison point for Section 3's leak-free weighted-conformal numbers: how close does transporting a TCGA-fit correction get to what you'd get calibrating directly on IPD Brain itself, with no domain classifier and no cross-cohort shift to correct for at all?

In [56]:
N_SPLITS = 50
CAL_FRAC = 0.5

def qhat_plain(cal_scores, alpha):
    n = len(cal_scores)
    q_level = min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)
    return float(np.quantile(cal_scores, q_level, method="higher"))

selfcal_rows = []
for cov, alpha in zip(TARGET_COVERAGES, ALPHAS):
    for seed in range(N_SPLITS):
        rng = np.random.default_rng(seed)
        idx = rng.permutation(len(eval_A_scorable))
        n_cal = int(CAL_FRAC * len(eval_A_scorable))
        cal_sub, ev_sub = eval_A_scorable.iloc[idx[:n_cal]], eval_A_scorable.iloc[idx[n_cal:]]
        cs = lac_score(np.where(cal_sub["label"] == 1, cal_sub["prob_mutant"], 1 - cal_sub["prob_mutant"]))
        q_hat = qhat_plain(cs, alpha)
        in0, in1 = prediction_sets(ev_sub["prob_mutant"].to_numpy(), q_hat)
        size = in0 + in1
        y = ev_sub["label"].to_numpy()
        covered = np.where(y == 1, in1, in0).astype(bool)
        selfcal_rows.append({"target_coverage": cov, "seed": seed, "n": len(size),
                              "coverage": float(covered.mean()),
                              "deferral_rate": float((size != 1).mean()),
                              "avg_size": float(size.mean()),
                              "n_empty": int((size == 0).sum()), "n_both": int((size == 2).sum())})

selfcal_df = pd.DataFrame(selfcal_rows)
selfcal_summary = selfcal_df.groupby("target_coverage").agg(
    coverage_mean=("coverage", "mean"), coverage_std=("coverage", "std"),
    deferral_mean=("deferral_rate", "mean"), deferral_std=("deferral_rate", "std"),
    avg_size_mean=("avg_size", "mean"), empty_mean=("n_empty", "mean"), both_mean=("n_both", "mean"),
).reset_index().sort_values("target_coverage", ascending=False)
selfcal_summary.to_csv(OUT_DIR / "ipd_self_calibration_ablation.csv", index=False)

print(f"IPD-self-calibrated plain conformal, {N_SPLITS} random {CAL_FRAC:.0%}/{1-CAL_FRAC:.0%} splits of Level A:")
display(selfcal_summary.round(3))

comparison = pd.DataFrame({
    "target_coverage": [0.90, 0.80, 0.70],
    "TCGA-cal, plain (coverage)": [
        results_df.query("tier=='Level A' and method=='plain' and target_coverage==@c")["coverage"].iloc[0]
        for c in [0.90, 0.80, 0.70]],
    "TCGA-cal, plain (deferral)": [
        sel_table.query("method=='plain' and target_coverage==@c")["defer_rate"].iloc[0]
        for c in [0.90, 0.80, 0.70]],
    "TCGA-cal, hoptimus-weighted (coverage)": [
        results_df.query("tier=='Level A' and method=='weighted_hoptimus' and target_coverage==@c")["coverage"].iloc[0]
        for c in [0.90, 0.80, 0.70]],
    "TCGA-cal, hoptimus-weighted (deferral)": [
        sel_table.query("method=='weighted_hoptimus' and target_coverage==@c")["defer_rate"].iloc[0]
        for c in [0.90, 0.80, 0.70]],
    "IPD-self-cal, plain (coverage)": selfcal_summary["coverage_mean"].to_numpy(),
    "IPD-self-cal, plain (deferral)": selfcal_summary["deferral_mean"].to_numpy(),
})
comparison.to_csv(OUT_DIR / "ipd_self_calibration_comparison.csv", index=False)
display(comparison.round(3))

IPD-self-calibrated plain conformal, 50 random 50%/50% splits of Level A:


,target_coverage,coverage_mean,coverage_std,deferral_mean,deferral_std,avg_size_mean,empty_mean,both_mean
2,0.9,0.916,0.036,0.385,0.108,1.385,0.00,45.04
1,0.8,0.818,0.044,0.046,0.048,1.027,1.12,4.32
0,0.7,0.718,0.058,0.136,0.065,0.864,15.92,0.00


,target_coverage,"TCGA-cal, plain (coverage)","TCGA-cal, plain (deferral)","TCGA-cal, hoptimus-weighted (coverage)","TCGA-cal, hoptimus-weighted (deferral)","IPD-self-cal, plain (coverage)","IPD-self-cal, plain (deferral)"
0,0.9,0.874,0.314,0.920,0.486,0.916,0.385
1,0.8,0.766,0.040,0.874,0.291,0.818,0.046
2,0.7,0.634,0.229,0.806,0.046,0.718,0.136


## 10b. Matched self-calibration: same scored patients as Section 3, with a paired test

Section 10's self-calibration benchmark (0.916/0.811/0.714) isn't directly
comparable to Section 3's TCGA-transported results: it's averaged over 50 random
50/50 splits of the full Level A pool (244 patients, ~122 scored per split), while
Section 3 scores a single fixed 182-patient `eval_score_df`. Different patients,
different sample sizes, no significance test between the two. This section fixes
that: calibrates on `A_weight_slice` (the same 62-patient, age-stratified,
reliable-label slice already reserved for weight-fitting in Section 2b, unused there
for its label since domain classifiers never see labels), and scores on the
identical `eval_score_df` used throughout Section 3. Same scored patients as the
H-optimus-weighted result, so a paired bootstrap test against it is now valid.

In [57]:
selfcal_scores = lac_score(np.where(
    A_weight_slice["label"] == 1, A_weight_slice["prob_mutant"], 1 - A_weight_slice["prob_mutant"]
))

selfcal_matched_rows, selfcal_covered_by_target = [], {}
test_rows = []
for cov, alpha in zip(TARGET_COVERAGES, ALPHAS):
    q_hat = qhat_plain(selfcal_scores, alpha)
    in0, in1 = prediction_sets(eval_score_df["prob_mutant"].to_numpy(), q_hat)
    size = in0 + in1
    y = eval_score_df["label"].to_numpy()
    covered = np.where(y == 1, in1, in0).astype(bool)
    selfcal_covered_by_target[cov] = covered
    ci_lo, ci_hi = coverage_bootstrap_ci(covered, seed=stable_seed("selfcal_matched", cov))
    selfcal_matched_rows.append({
        "target_coverage": cov, "n_cal": len(A_weight_slice), "n_scored": len(eval_score_df),
        "coverage": float(covered.mean()), "ci_lo": ci_lo, "ci_hi": ci_hi,
        "deferral_rate": float((size != 1).mean()), "n_empty": int((size == 0).sum()),
    })

    cal_w_hopt, score_w_hopt, _ = WEIGHTS_BY_TIER_SCHEME["Level A"]["weighted_hoptimus"]
    covered_hopt = covered_bool(eval_score_df, cal_scores_true, cal_w_hopt, score_w_hopt, alpha)
    obs, lo, hi, p = paired_diff_test(covered_hopt, covered, seed=stable_seed("selfcal_vs_hoptimus", cov))
    test_rows.append({
        "target_coverage": cov, "hoptimus_coverage": float(covered_hopt.mean()),
        "selfcal_matched_coverage": float(covered.mean()),
        "delta_hoptimus_minus_selfcal": obs, "ci_lo": lo, "ci_hi": hi, "p": p,
    })

selfcal_matched_df = pd.DataFrame(selfcal_matched_rows)
test_df = pd.DataFrame(test_rows)

selfcal_matched_df.to_csv(OUT_DIR / "ipd_self_calibration_matched.csv", index=False)
test_df.to_csv(OUT_DIR / "ipd_self_calibration_vs_hoptimus_test.csv", index=False)

print("Matched self-calibration (cal on A_weight_slice, scored on eval_score_df):")
display(selfcal_matched_df.round(4))
print("\nPaired test: H-optimus-weighted vs. matched self-calibration (same eval_score_df):")
display(test_df.round(4))


Matched self-calibration (cal on A_weight_slice, scored on eval_score_df):


,target_coverage,n_cal,n_scored,coverage,ci_lo,ci_hi,deferral_rate,n_empty
0,0.9,58,175,0.8457,0.7886,0.8971,0.1771,0
1,0.8,58,175,0.7600,0.6971,0.8229,0.0457,8
2,0.7,58,175,0.5543,0.4800,0.6286,0.3371,59



Paired test: H-optimus-weighted vs. matched self-calibration (same eval_score_df):


,target_coverage,hoptimus_coverage,selfcal_matched_coverage,delta_hoptimus_minus_selfcal,ci_lo,ci_hi,p
0,0.9,0.9200,0.8457,0.0743,0.0399,0.1143,0.001
1,0.8,0.8743,0.7600,0.1143,0.0686,0.1600,0.001
2,0.7,0.8057,0.5543,0.2514,0.1886,0.3143,0.001


## Saved artifacts

In [58]:
for f in sorted(OUT_DIR.glob("*")):
    print(f)

/cs/student/project_msc/2025/aibh/mpapageo/outputs/shift_weighted_conformal_ipd_brain_extension/coverage_results.csv
/cs/student/project_msc/2025/aibh/mpapageo/outputs/shift_weighted_conformal_ipd_brain_extension/domain_classifier_ess.csv
/cs/student/project_msc/2025/aibh/mpapageo/outputs/shift_weighted_conformal_ipd_brain_extension/fullfeature_coverage.csv
/cs/student/project_msc/2025/aibh/mpapageo/outputs/shift_weighted_conformal_ipd_brain_extension/ipd_self_calibration_ablation.csv
/cs/student/project_msc/2025/aibh/mpapageo/outputs/shift_weighted_conformal_ipd_brain_extension/ipd_self_calibration_comparison.csv
/cs/student/project_msc/2025/aibh/mpapageo/outputs/shift_weighted_conformal_ipd_brain_extension/ipd_self_calibration_matched.csv
/cs/student/project_msc/2025/aibh/mpapageo/outputs/shift_weighted_conformal_ipd_brain_extension/ipd_self_calibration_matched_AB.csv
/cs/student/project_msc/2025/aibh/mpapageo/outputs/shift_weighted_conformal_ipd_brain_extension/ipd_self_calibration_